In [39]:
# Importa a biblioteca pandas para manipulação de dados em DataFrames
import pandas as pd

# Importa a biblioteca numpy para operações numéricas
import numpy as np

# Importa a função para modelos lineares do statsmodels usando fórmulas
import statsmodels.formula.api as smf

# Importa seaborn para visualização de dados estatísticos
import seaborn as sns

# Importa o módulo pyplot do matplotlib para criação de gráficos
import matplotlib.pyplot as plt

# Importa a API principal do statsmodels para análise estatística
import statsmodels.api as sm

# Importa função para dividir os dados em treino e teste
from sklearn.model_selection import train_test_split

# Importa a função para cálculo do VIF (Variance Inflation Factor)
from statsmodels.stats.outliers_influence import variance_inflation_factor as vif
from statsmodels.tools.tools import add_constant

import unicodedata


In [40]:
df = pd.read_csv('./Dataset/previsao_de_renda_II.csv')
df.loc[df['tempo_emprego'].isna(), 'tempo_emprego'] = df.tempo_emprego.mean()

## Multicolinearidade

Avalie se há questões relacionadas a multicolinearidade através de pelo menos:

- Matriz de correlação de Spearman
- VIF

In [41]:
# Função para criar dataframe VIF e realizar a filtragem de variáveis que possuirem um valor acima do estipulado nos parâmetros
def vif_filter(X, limite=10):
    """
    Filtra variáveis com alto fator de inflação de variância (VIF) de um DataFrame.

    Parâmetros:
    X (pd.DataFrame): DataFrame contendo apenas variáveis numéricas independentes.
    limite (float): Valor limite de VIF para remoção de variáveis (padrão=10).

    Retorna:
    X_filtrado (pd.DataFrame): DataFrame com variáveis remanescentes após filtragem.
    removed_features (list): Lista de tuplas (variável, VIF) removidas.
    remaining_vif (pd.DataFrame): DataFrame com VIF das variáveis remanescentes.
    """
    X_filtrado = X.copy()
    removed_features = []
    vif_scores = {}

    while True:
        # Adiciona constante para cálculo do VIF
        X_with_const = add_constant(X_filtrado)

        # Calcula o VIF para cada variável (incluindo a constante)
        vif_data = pd.DataFrame()
        vif_data["feature"] = X_with_const.columns
        vif_data["VIF"] = [vif(X_with_const.values, i)
                           for i in range(X_with_const.shape[1])]

        # Remove a constante da análise
        vif_data = vif_data[vif_data['feature'] != 'const']

        # Salva os VIFs calculados
        for _, row in vif_data.iterrows():
            vif_scores[row['feature']] = row['VIF']

        # Verifica o maior VIF
        max_vif = vif_data['VIF'].max()
        if max_vif <= limite:
            break

        # Remove a variável com maior VIF
        feature_to_remove = vif_data.loc[vif_data['VIF'].idxmax(), 'feature']
        removed_features.append((feature_to_remove, max_vif))
        X_filtrado = X_filtrado.drop(columns=[feature_to_remove])

        # Se todas as variáveis forem removidas, lança erro
        if X_filtrado.shape[1] == 0:
            raise ValueError("Todas as variáveis foram removidas - limite pode estar muito baixo")

    # Monta DataFrame final com VIF das variáveis remanescentes
    remaining_vif = vif_data[vif_data['feature'].isin(X_filtrado.columns)]
    remaining_vif = remaining_vif.sort_values('VIF', ascending=False)

    return X_filtrado, removed_features, remaining_vif



# Função para remover caracteres especiais e adicionar "_" nos espaços
def padronizar_nome(var):
    """
    Remove caracteres especiais de uma string e substitui espaços por underline.

    Parâmetros:
    var (str): Nome da variável a ser padronizado.

    Retorna:
    str: Nome padronizado, sem acentos e com espaços substituídos por "_".
    """
    var = unicodedata.normalize('NFKD', var).encode('ASCII', 'ignore').decode('ASCII')
    return var.replace(' ', '_')

In [42]:
# Obtém a lista de todas as colunas do DataFrame df
variaveis = df.columns.to_list()

# Remove as colunas 'data_ref' e 'index' da lista de variáveis, pois não serão usadas nas análises
variaveis = [x for x in variaveis if x not in ['data_ref', 'index']]

In [43]:
# Calcula a matriz de correlação de Spearman apenas para as variáveis numéricas do dataframe 'df'
correlacao_spearmen = df[variaveis].select_dtypes(include=['float64', 'int64']).corr(method='spearman')

# Substitui os valores da diagonal principal por NaN para facilitar a visualização (evita destacar a correlação perfeita de uma variável consigo mesma)
np.fill_diagonal(correlacao_spearmen.values, np.nan)
        

In [44]:
# Aplica o destaque (highlight) ao maior valor de cada linha da matriz de correlação de Spearman,
# usando a cor preta para facilitar a visualização dos pares de variáveis com maior correlação em cada linha.
correlacao_spearmen.style.highlight_max(axis=1, color='black')

,qtd_filhos,idade,tempo_emprego,qt_pessoas_residencia,renda
qtd_filhos,nan,-0.415151,-0.089260,0.828600,-0.019957
idade,-0.415151,nan,0.300547,-0.350006,0.107999
tempo_emprego,-0.089260,0.300547,nan,-0.058139,0.501354
qt_pessoas_residencia,0.828600,-0.350006,-0.058139,nan,-0.008260
renda,-0.019957,0.107999,0.501354,-0.008260,nan


In [45]:
# Cria variáveis dummies para as variáveis categóricas presentes em 'variaveis' do DataFrame 'df'
# drop_first=True remove a primeira categoria de cada variável para evitar multicolinearidade (dummy variable trap)
# dtype=int garante que as colunas criadas sejam do tipo inteiro
previsao_renda_dummies_numericas = pd.get_dummies(
    data=df[variaveis], drop_first=True, dtype=int
)

# Exibe o DataFrame resultante com as variáveis numéricas e dummies
previsao_renda_dummies_numericas

,qtd_filhos,idade,tempo_emprego,qt_pessoas_residencia,renda,sexo_M,posse_de_veiculo_S,posse_de_imovel_S,tipo_renda_Bolsista,tipo_renda_Empresário,...,educacao_Superior incompleto,estado_civil_Separado,estado_civil_Solteiro,estado_civil_União,estado_civil_Viúvo,tipo_residencia_Casa,tipo_residencia_Com os pais,tipo_residencia_Comunitário,tipo_residencia_Estúdio,tipo_residencia_Governamental
0,0,47,16.717808,2.0,11138.14,1,1,0,0,1,...,0,0,0,0,0,1,0,0,0,0
1,0,30,9.600000,2.0,2424.81,1,1,0,0,0,...,1,0,0,0,0,1,0,0,0,0
2,0,28,8.208219,2.0,13749.66,1,1,0,0,0,...,0,0,0,0,0,1,0,0,0,0
3,2,44,1.301370,4.0,2361.84,1,0,1,0,0,...,0,0,0,0,0,1,0,0,0,0
4,2,33,1.254795,4.0,790.78,0,0,1,0,0,...,0,0,0,0,0,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
749995,1,29,8.665753,3.0,2930.40,0,0,1,0,0,...,0,0,0,0,0,1,0,0,0,0
749996,0,65,7.746165,2.0,4084.37,0,0,1,0,0,...,0,0,0,0,0,1,0,0,0,0
749997,0,33,10.969863,2.0,4339.66,0,0,1,0,0,...,0,0,0,0,0,1,0,0,0,0
749998,1,28,8.219178,3.0,9159.49,0,1,1,0,1,...,0,0,0,1,0,1,0,0,0,0


In [46]:
# Renomeia as colunas do DataFrame aplicando a função de padronização de nomes
previsao_renda_dummies_numericas = previsao_renda_dummies_numericas.rename(columns=padronizar_nome)

# Cria uma lista com todas as variáveis independentes, exceto 'renda'
variaveis_independentes = [var for var in previsao_renda_dummies_numericas.columns.to_list() if var != 'renda']

# Monta a string da fórmula para o modelo de regressão linear
formula = "renda ~ " + " + ".join(variaveis_independentes)

# Ajusta o modelo de regressão linear com a fórmula e os dados
r1 = smf.ols(
    formula,
    data=previsao_renda_dummies_numericas
).fit()

In [47]:
r1.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                  renda   R-squared:                       0.218
Model:                            OLS   Adj. R-squared:                  0.218
Method:                 Least Squares   F-statistic:                     8736.
Date:                Tue, 08 Jul 2025   Prob (F-statistic):               0.00
Time:                        22:36:52   Log-Likelihood:            -9.5308e+06
No. Observations:              750000   AIC:                         1.906e+07
Df Residuals:                  749975   BIC:                         1.906e+07
Df Model:                          24                                         
Covariance Type:            nonrobust                                         
=================================================================================================
                                    coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------------
Intercept                     -3.729e+04   3713.392    -10.043      0.000   -4.46e+04      -3e+04
qtd_filhos                    -1216.8220   1750.251     -0.695      0.487   -4647.257    2213.613
idade                           275.3811     12.087     22.783      0.000     251.691     299.071
tempo_emprego                  6687.2758     16.271    410.992      0.000    6655.385    6719.167
qt_pessoas_residencia           300.1166   1746.538      0.172      0.864   -3123.040    3723.274
sexo_M                        -2721.8917    217.934    -12.489      0.000   -3149.036   -2294.747
posse_de_veiculo_S             -816.4765    207.233     -3.940      0.000   -1222.647    -410.306
posse_de_imovel_S              5587.9791    202.780     27.557      0.000    5190.536    5985.422
tipo_renda_Bolsista           -3.312e+04   4124.059     -8.031      0.000   -4.12e+04    -2.5e+04
tipo_renda_Empresario          2323.9973    233.848      9.938      0.000    1865.663    2782.332
tipo_renda_Pensionista        -2.153e+04    333.742    -64.502      0.000   -2.22e+04   -2.09e+04
tipo_renda_Servidor_publico    -262.1285    344.101     -0.762      0.446    -936.555     412.298
educacao_Medio                -2667.8349    876.894     -3.042      0.002   -4386.519    -949.151
educacao_Pos_graduacao         6300.1949   2958.800      2.129      0.033     501.045    1.21e+04
educacao_Superior_completo    -1226.8484    884.768     -1.387      0.166   -2960.965     507.268
educacao_Superior_incompleto   4132.4120    991.224      4.169      0.000    2189.645    6075.179
estado_civil_Separado          1585.2603   1792.339      0.884      0.376   -1927.666    5098.187
estado_civil_Solteiro          9028.3729   1758.174      5.135      0.000    5582.410    1.25e+04
estado_civil_Uniao              904.8148    352.209      2.569      0.010     214.498    1595.132
estado_civil_Viuvo             -632.0469   1811.931     -0.349      0.727   -4183.372    2919.278
tipo_residencia_Casa          -1605.4287    793.217     -2.024      0.043   -3160.109     -50.749
tipo_residencia_Com_os_pais    2207.3627    899.601      2.454      0.014     444.175    3970.550
tipo_residencia_Comunitario    3022.2969   1604.861      1.883      0.060    -123.179    6167.773
tipo_residencia_Estudio       -4627.8536   1341.766     -3.449      0.001   -7257.671   -1998.036
tipo_residencia_Governamental -5239.4825    945.534     -5.541      0.000   -7092.698   -3386.266
==============================================================================
Omnibus:                  1840375.888   Durbin-Watson:                   1.928
Prob(Omnibus):                  0.000   Jarque-Bera (JB):      49429691063.961
Skew:                          25.959   Prob(JB):                         0.00
Kurtosis:              

In [48]:
previsao_renda_vif_ajustado, variaveis_removidas, vif_frame = vif_filter(previsao_renda_dummies_numericas.drop(columns=['renda']), 5)

In [54]:
# Monta a string da fórmula para o modelo de regressão linear
formula = "I(np.log(renda + 1)) ~ " + " + ".join(vif_frame.feature)

# Ajusta o modelo de regressão linear com a fórmula e os dados
r2 = smf.ols(
    formula,
    data=previsao_renda_dummies_numericas
).fit()

In [50]:
r2.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                             OLS Regression Results                             
================================================================================
Dep. Variable:     I(np.log(renda + 1))   R-squared:                       0.393
Model:                              OLS   Adj. R-squared:                  0.393
Method:                   Least Squares   F-statistic:                 2.315e+04
Date:                  Tue, 08 Jul 2025   Prob (F-statistic):               0.00
Time:                          22:38:24   Log-Likelihood:            -1.0643e+06
No. Observations:                750000   AIC:                         2.129e+06
Df Residuals:                    749978   BIC:                         2.129e+06
Df Model:                            21                                         
Covariance Type:              nonrobust                                         
=================================================================================================
                                    coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------------
Intercept                         7.7167      0.007   1088.875      0.000       7.703       7.731
idade                             0.0035      0.000     23.276      0.000       0.003       0.004
tipo_renda_Pensionista           -0.2963      0.004    -71.027      0.000      -0.304      -0.288
sexo_M                            0.1269      0.003     46.556      0.000       0.122       0.132
qtd_filhos                        0.0047      0.002      2.740      0.006       0.001       0.008
posse_de_veiculo_S                0.0151      0.003      5.819      0.000       0.010       0.020
tempo_emprego                     0.1289      0.000    633.370      0.000       0.129       0.129
tipo_renda_Empresario             0.1526      0.003     52.161      0.000       0.147       0.158
tipo_renda_Servidor_publico       0.1461      0.004     33.919      0.000       0.138       0.154
estado_civil_Viuvo                0.0074      0.006      1.227      0.220      -0.004       0.019
estado_civil_Solteiro             0.0094      0.004      2.567      0.010       0.002       0.017
educacao_Superior_completo       -0.0063      0.003     -2.536      0.011      -0.011      -0.001
tipo_residencia_Com_os_pais       0.0313      0.006      5.404      0.000       0.020       0.043
posse_de_imovel_S                 0.1867      0.003     73.611      0.000       0.182       0.192
educacao_Superior_incompleto     -0.0198      0.006     -3.236      0.001      -0.032      -0.008
estado_civil_Separado             0.0305      0.005      5.995      0.000       0.021       0.040
estado_civil_Uniao                0.0017      0.004      0.379      0.705      -0.007       0.010
tipo_residencia_Governamental    -0.0136      0.007     -2.032      0.042      -0.027      -0.000
tipo_residencia_Estudio          -0.0394      0.014     -2.881      0.004      -0.066      -0.013
tipo_residencia_Comunitario      -0.0411      0.018     -2.343      0.019      -0.076      -0.007
tipo_renda_Bolsista              -0.1580      0.052     -3.061      0.002      -0.259      -0.057
educacao_Pos_graduacao            0.0639      0.035      1.805      0.071      -0.005       0.133
==============================================================================
Omnibus:                     3983.688   Durbin-Watson:                   1.036
Prob(Omnibus):                  0.000   Jarque-Bera (JB):             2868.923
Skew:                          -0.022   Prob(JB):                         0.00
Kurtosis:                       2.700   Cond. No.                     2.05e+03
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 2.

In [55]:
previsao_renda_vif_ajustado.select_dtypes(include=['float64', 'int64'])

,qtd_filhos,idade,tempo_emprego
0,0,47,16.717808
1,0,30,9.600000
2,0,28,8.208219
3,2,44,1.301370
4,2,33,1.254795
...,...,...,...
749995,1,29,8.665753
749996,0,65,7.746165
749997,0,33,10.969863
749998,1,28,8.219178
